In [1]:
import arcpy
import json
from collections import defaultdict

In [2]:
fc = r"D:\Documents\ArcGIS\Projects\ProyectoGrado\ProyectoGrado.gdb\lineas_vision_final_completo"

In [3]:
data = defaultdict(lambda: defaultdict(lambda: {
    "lenght": 0,
    "visible": 0
}))

In [4]:
fields = ["SOURCEOID", "OID_OBSERV", "VisCode", "TarIsVis", "Shape_Length", "OID_TARGET", "AZIMUTH", "VERT_ANGLE", "Shape_Length_1", "OBSTR_MPID"]

In [5]:
with arcpy.da.SearchCursor(fc, fields) as cursor: 
    for sourceoid, oid_observ, viscode, tarisvis, shape_length, oid_target, azimuth, vert_angle, shape_length_1, obstr_mpid in cursor: 
        data[oid_observ][oid_target]["lenght"] = shape_length_1 
        data[oid_observ][oid_target]["azimuth"] = azimuth 
        data[oid_observ][oid_target]["vert_angle"] = vert_angle 
        data[oid_observ][oid_target]["obstr_mpid"] = obstr_mpid 
        
        if tarisvis == 1: 
            data[oid_observ][oid_target]["visible"] = shape_length_1 
        elif viscode == 1: 
            data[oid_observ][oid_target]["visible"] += shape_length

In [6]:
camaras = []
for oid_observer, sources in data.items():
    total_visibles = 0
    total_lineas = 0
    total_targets = 0
    lista_sources = []

    for sourceoid, vals in sources.items():
        total = vals["lenght"]
        visibles = vals["visible"]
        
        porcentaje = (visibles / total) * 100 if total > 0 else 0
        
        if porcentaje >= 90:
            lista_sources.append({
                "target_id": sourceoid,
                "azimuth": vals["azimuth"],
                "vert_angle": vals["vert_angle"],
                "obstr_mpid": vals["obstr_mpid"], 
                "length": total,
                "visible": visibles,
                "porcentaje": round(porcentaje, 2)
            })

        if porcentaje >= 90:
            total_targets += 1

        total_visibles += visibles
        total_lineas += total
        
    camaras.append({
        "OID_OBSERVER": oid_observer,
        "porcentaje_final": round(
            (total_visibles / total_lineas) * 100, 2
        ),
        "total_length": total_lineas,
        "total_targets": total_targets,
        "sources": lista_sources
    })

In [7]:
def seleccionar_camaras_greedy(camaras, umbral=0.95):
    # 1️⃣ Crear el set de todos los targets
    todos_targets = set()
    for cam in camaras:
        todos_targets.update([s["target_id"] for s in cam["sources"]])
    
    # Objetivo: cubrir al menos umbral * total_targets
    total_a_cubrir = int(len(todos_targets) * umbral)
    
    # 2️⃣ Inicializar variables
    targets_cubiertos = set()
    camaras_seleccionadas = []
    
    # 3️⃣ Loop greedy
    while len(targets_cubiertos) < total_a_cubrir:
        mejor_camara = None
        targets_nuevos_max = 0
        
        for cam in camaras:
            # Targets que esta cámara puede cubrir y que aún no estén cubiertos
            targets_nuevos = set(s["target_id"] for s in cam["sources"]) - targets_cubiertos
            if len(targets_nuevos) > targets_nuevos_max:
                targets_nuevos_max = len(targets_nuevos)
                mejor_camara = cam
        
        if not mejor_camara:
            # Ya no quedan cámaras que agreguen nuevos targets
            break
        
        # Añadir cámara seleccionada
        camaras_seleccionadas.append(mejor_camara)
        # Marcar targets cubiertos
        targets_cubiertos.update(s["target_id"] for s in mejor_camara["sources"])
        # Opcional: remover la cámara de la lista para no reconsiderarla
        camaras.remove(mejor_camara)
        
        print(f"Seleccionada cámara {mejor_camara['OID_OBSERVER']}, "
              f"cubriendo {targets_nuevos_max} targets nuevos, "
              f"{len(targets_cubiertos)}/{total_a_cubrir} cubiertos")
    
    print(f"\nTotal cámaras seleccionadas: {len(camaras_seleccionadas)}")
    print(f"Porcentaje de targets cubiertos: {len(targets_cubiertos)/len(todos_targets)*100:.2f}%")
    
    return camaras_seleccionadas

In [8]:
seleccionadas = seleccionar_camaras_greedy(camaras, umbral=0.95)

Seleccionada cámara 14329, cubriendo 584 targets nuevos, 584/12172 cubiertos
Seleccionada cámara 18265, cubriendo 490 targets nuevos, 1074/12172 cubiertos
Seleccionada cámara 17678, cubriendo 374 targets nuevos, 1448/12172 cubiertos
Seleccionada cámara 17952, cubriendo 351 targets nuevos, 1799/12172 cubiertos
Seleccionada cámara 6303, cubriendo 341 targets nuevos, 2140/12172 cubiertos
Seleccionada cámara 17415, cubriendo 313 targets nuevos, 2453/12172 cubiertos
Seleccionada cámara 17382, cubriendo 284 targets nuevos, 2737/12172 cubiertos
Seleccionada cámara 10433, cubriendo 279 targets nuevos, 3016/12172 cubiertos
Seleccionada cámara 18959, cubriendo 273 targets nuevos, 3289/12172 cubiertos
Seleccionada cámara 4842, cubriendo 256 targets nuevos, 3545/12172 cubiertos
Seleccionada cámara 12168, cubriendo 252 targets nuevos, 3797/12172 cubiertos
Seleccionada cámara 17900, cubriendo 237 targets nuevos, 4034/12172 cubiertos
Seleccionada cámara 17435, cubriendo 230 targets nuevos, 4264/12172

In [9]:
oids = [item["OID_OBSERVER"] for item in seleccionadas]
oids_str = ",".join(map(str, oids))
sql = f"OID_OBSERVER IN ({oids_str})"
sql

'OID_OBSERVER IN (14329,18265,17678,17952,6303,17415,17382,10433,18959,4842,12168,17900,17435,17670,1208,5792,17300,7790,15495,2354,2731,15812,2284,17508,18883,18164,18916,17521,3668,18286,4457,17723,17795,1914,12822,17442,12867,14735,18077,18157,928,6811,18227,5165,15181,17892,1008,12912,8867,9714,4873,18653,3939,16310,3753,12924,18623,4059,7047,5608,6822,839,3690,13393,17977,17568,14779,9016,1096,18611,4584,16570,11919,5556,14016,14378,7743,16435,10772,11723,18134,18556,7094,5658,7235,5946,8042,15805,18543,7535,12324,16029,1694,6728,8606,13253,16999,9184,9435,18355,10434,18102,2811,7497,9209,700,18772,3531,5951,7063,11435,13903,17789,7155,12183,14634,4838,5102,3268,9286,10349,18087,18588,272,474,718,18296,666,3815,8261,12092,14306,16302,2482)'

In [13]:
import random
def encontrar_punto_quiebre(camaras_seleccionadas, umbral_critico=70.0, repeticiones=50):
    """
    Aumenta n (cámaras perdidas) hasta que la cobertura promedio cae bajo el umbral.
    Retorna el 'Margen de Seguridad' (n máximo soportado).
    """
    total_targets_ref = set()
    for cam in camaras_seleccionadas:
        total_targets_ref.update([s["target_id"] for s in cam["sources"]])
    
    total_inicial = len(total_targets_ref)
    n_perdidas = 1
    
    print(f"--- Buscando Punto de Quiebre (Umbral: {umbral_critico}%) ---")
    
    while n_perdidas < len(camaras_seleccionadas):
        coberturas_iteracion = []
        
        for _ in range(repeticiones):
            # Simulación aleatoria
            restantes = random.sample(camaras_seleccionadas, len(camaras_seleccionadas) - n_perdidas)
            targets_vivos = set()
            for c in restantes:
                targets_vivos.update([s["target_id"] for s in c["sources"]])
            
            porcentaje = (len(targets_vivos) / total_inicial) * 100
            coberturas_iteracion.append(porcentaje)
        
        promedio_actual = sum(coberturas_iteracion) / repeticiones
        
        if n_perdidas % 5 == 0: # Para no saturar la consola, imprime cada 5
            print(f"Perdiendo {n_perdidas} cámaras: Cobertura promedio {promedio_actual:.2f}%")
            
        if promedio_actual < umbral_critico:
            margen_seguridad = n_perdidas - 1
            print(f"\n¡PUNTO DE QUIEBRE ALCANZADO!")
            print(f"Tu Margen de Seguridad es de {margen_seguridad} cámaras.")
            print(f"Con {n_perdidas} cámaras perdidas, la cobertura cae a {promedio_actual:.2f}%")
            return margen_seguridad
        
        n_perdidas += 1

    return len(camaras_seleccionadas)

# --- Ejecución ---
margen = encontrar_punto_quiebre(seleccionadas, umbral_critico=70.0)

--- Buscando Punto de Quiebre (Umbral: 70.0%) ---
Perdiendo 5 cámaras: Cobertura promedio 98.28%
Perdiendo 10 cámaras: Cobertura promedio 96.57%
Perdiendo 15 cámaras: Cobertura promedio 94.62%
Perdiendo 20 cámaras: Cobertura promedio 92.56%
Perdiendo 25 cámaras: Cobertura promedio 90.27%
Perdiendo 30 cámaras: Cobertura promedio 88.29%
Perdiendo 35 cámaras: Cobertura promedio 85.92%
Perdiendo 40 cámaras: Cobertura promedio 83.27%
Perdiendo 45 cámaras: Cobertura promedio 80.23%
Perdiendo 50 cámaras: Cobertura promedio 77.19%
Perdiendo 55 cámaras: Cobertura promedio 74.59%
Perdiendo 60 cámaras: Cobertura promedio 71.56%

¡PUNTO DE QUIEBRE ALCANZADO!
Tu Margen de Seguridad es de 61 cámaras.
Con 62 cámaras perdidas, la cobertura cae a 69.86%
